In [ ]:
import pandas as pd
filepath= r"C:\Users\Haritha\Downloads\Uber_Eats_data.csv"
a=pd.read_csv(filepath)
display(a.head())



,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),listed_in(type),listed_in(city)
0,Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,Buffet,Banashankari
1,Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,Buffet,Banashankari
2,San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,Buffet,Banashankari
3,Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,Buffet,Banashankari
4,Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,Buffet,Banashankari


In [3]:
#to drop duplicates
a.drop_duplicates(inplace=True)

In [4]:
#to check null values
print(a.isnull().sum())

name                           0
online_order                   0
book_table                     0
rate                           0
votes                          0
phone                          0
location                       0
rest_type                      0
dish_liked                     0
cuisines                       0
approx_cost(for two people)    0
listed_in(type)                0
listed_in(city)                0
dtype: int64


In [5]:
import numpy as np

# Remove '/5' from the 'rate' column and convert it to a numeric type
a['rate'] = a['rate'].astype(str).str.replace('/5', '', regex=False)
a['rate'] = pd.to_numeric(a['rate'], errors='coerce')

# Fill any NaN values that resulted from the conversion with the mean of the column
a['rate'].fillna(a['rate'].mean(), inplace=True)

# Verify the changes
print("Data type of 'rate' after cleaning:", a['rate'].dtype)
print("Number of NaN values in 'rate' after cleaning:", a['rate'].isna().sum())
display(a.head())

Data type of 'rate' after cleaning: float64
Number of NaN values in 'rate' after cleaning: 0


,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),listed_in(type),listed_in(city)
0,Jalsa,Yes,Yes,4.1,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,Buffet,Banashankari
1,Spice Elephant,Yes,No,4.1,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,Buffet,Banashankari
2,San Churro Cafe,Yes,No,3.8,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,Buffet,Banashankari
3,Addhuri Udupi Bhojana,No,No,3.7,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,Buffet,Banashankari
4,Grand Village,No,No,3.8,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,Buffet,Banashankari


In [44]:

import pymysql

try:
    conn = pymysql.connect(
        host="gateway01.ap-southeast-1.prod.aws.tidbcloud.com",
        port=4000,
        user="2LQdqxSJhDf5xi1.root",
        password="Av6yG6tlEREJUg0M",
        database="eatdata",
        # Use a raw string (r"") and the 'ca' key
        ssl={'ca': r"C:\Users\Haritha\Downloads\isrgrootx1 (1).pem"},
        autocommit=True
    )
    print("Connection successful!")
    cursor = conn.cursor()

except pymysql.MySQLError as e:
    print(f"Error connecting to TiDB: {e}")






Connection successful!


In [8]:
cursor.execute("DROP TABLE IF EXISTS eatdata.ubereats")

# Create the table 'ubereats' with the correct schema, including Normalizedrate and scaled_approx_cost
create_table_sql_ubereats = """
CREATE TABLE ubereats (
    name VARCHAR(255),
    online_order VARCHAR(3),
    book_table VARCHAR(3),
    rate FLOAT,
    votes INT,
    phone VARCHAR(255),
    location VARCHAR(255),
    rest_type VARCHAR(255),
    dish_liked TEXT,
    cuisines TEXT,
    approx_cost_for_two_people FLOAT,
    listed_in_type VARCHAR(255),
    listed_in_city VARCHAR(255),
    Normalizedrate FLOAT,
    scaled_approx_cost FLOAT
);
"""
cursor.execute(create_table_sql_ubereats)
print("Table 'ubereats' created successfully.")

Table 'ubereats' created successfully.


In [9]:
import numpy as np

# Create a temporary copy of 'a' to perform renames
a_for_sql_prep = a.copy()

# Rename columns in the temporary DataFrame to match SQL-friendly names
a_for_sql_prep = a_for_sql_prep.rename(columns={
    'approx_cost(for two people)': 'approx_cost_for_two_people',
    'listed_in(type)': 'listed_in_type',
    'listed_in(city)': 'listed_in_city'
})

# Clean and convert 'approx_cost_for_two_people' to float, handling commas
a_for_sql_prep['approx_cost_for_two_people'] = a_for_sql_prep['approx_cost_for_two_people'].astype(str).str.replace(',', '', regex=False)
a_for_sql_prep['approx_cost_for_two_people'] = pd.to_numeric(a_for_sql_prep['approx_cost_for_two_people'], errors='coerce')

# Explicitly select the columns for insertion that match the ubereats table schema
df_for_sql = a_for_sql_prep[[
    'name', 'online_order', 'book_table', 'rate', 'votes', 'phone',
    'location', 'rest_type', 'dish_liked', 'cuisines',
    'approx_cost_for_two_people', 'listed_in_type', 'listed_in_city'
]]

# Replace NaN values with None for SQL compatibility
df_for_sql_cleaned = df_for_sql.replace({np.nan: None})

# Define the SQL INSERT statement with the correct columns
insert_sql = """
INSERT INTO ubereats (
    name, online_order, book_table, rate, votes, phone, location, rest_type, dish_liked, cuisines,
    approx_cost_for_two_people, listed_in_type, listed_in_city
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

# Prepare data for insertion as a list of tuples
data_to_insert = df_for_sql_cleaned.values.tolist()

# Execute batch insert
cursor.executemany(insert_sql, data_to_insert)

# Commit the transaction
conn.commit()

print(f"{cursor.rowcount} rows inserted successfully into 'ubereats' table.")


23158 rows inserted successfully into 'ubereats' table.


In [11]:
query1 = """
SELECT
    location,
    AVG(rate) AS average_rating
FROM
    ubereats
GROUP BY
    location
ORDER BY
    average_rating DESC
LIMIT 10;
"""
cursor.execute(query1)
results = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]
df_avg_ratings = pd.DataFrame(results, columns=column_names)
print("Top 10 pBangalore Locations with Highest Average Restaurant Ratings:",results)
display(df_avg_ratings)




Top 10 pBangalore Locations with Highest Average Restaurant Ratings: (('Lavelle Road', 4.192533980127913), ('Koramangala 5th Block', 4.148312025332424), ('Sankey Road', 4.1058823361116294), ('Koramangala 3rd Block', 4.101242192783711), ('Cunningham Road', 4.100300258344358), ('St. Marks Road', 4.0990131575810285), ('Koramangala 2nd Block', 4.073333263397217), ('Sadashiv Nagar', 4.055263218126799), ('Residency Road', 4.051363662698052), ('Church Street', 4.045454559118851))


,location,average_rating
0,Lavelle Road,4.192534
1,Koramangala 5th Block,4.148312
2,Sankey Road,4.105882
3,Koramangala 3rd Block,4.101242
4,Cunningham Road,4.100300
5,St. Marks Road,4.099013
6,Koramangala 2nd Block,4.073333
7,Sadashiv Nagar,4.055263
8,Residency Road,4.051364
9,Church Street,4.045455


In [12]:
query2 = """
SELECT
    location,
    COUNT(name) AS restaurant_count
FROM
    ubereats
GROUP BY
    location
ORDER BY
    restaurant_count DESC
LIMIT 20;
"""
cursor.execute(query2)
results = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]
df_locations = pd.DataFrame(results, columns=column_names)
print("Top 20 Bangalore Locations Over-saturated with Restaurants:")
display(df_locations)

Top 20 Bangalore Locations Over-saturated with Restaurants:


,location,restaurant_count
0,Koramangala 5th Block,1782
1,BTM,1454
2,Indiranagar,1346
3,HSR,1161
4,Jayanagar,1037
5,JP Nagar,1016
6,Whitefield,824
7,Koramangala 7th Block,723
8,Koramangala 6th Block,718
9,Marathahalli,680


In [13]:
import pandas as pd

query3 = """
SELECT
    online_order,
    AVG(rate) AS average_rating
FROM
    ubereats
GROUP BY
    online_order;
"""


cursor.execute(query3)
results = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]

# Create a DataFrame from the results
df_online_order_ratings = pd.DataFrame(results, columns=column_names)

# Display the results
print("Average Restaurant Ratings by Online Ordering Availability:")
display(df_online_order_ratings)

Average Restaurant Ratings by Online Ordering Availability:


,online_order,average_rating
0,Yes,3.893465
1,No,3.929231


In [14]:

# SQL query to compare average ratings for table booking vs. no table booking
query4 = """
SELECT
    book_table,
    AVG(rate) AS average_rating
FROM
    ubereats
GROUP BY
    book_table;
"""

# Execute the query and fetch results
cursor.execute(query4)
results = cursor.fetchall()

# Get column names from cursor description
column_names = [desc[0] for desc in cursor.description]

# Create a DataFrame from the results
df_book_table_ratings = pd.DataFrame(results, columns=column_names)

# Display the results
print("Average Restaurant Ratings by Table Booking Availability:")
display(df_book_table_ratings)

Average Restaurant Ratings by Table Booking Availability:


,book_table,average_rating
0,No,3.81357
1,Yes,4.15683


In [15]:
import pandas as pd
import numpy as np

# SQL query to fetch approximate cost and rate for all restaurants
query_cost_ratings5 = """
SELECT
    approx_cost_for_two_people,
    rate
FROM
    ubereats
WHERE rate IS NOT NULL;
"""

# Execute the query and fetch results
cursor.execute(query_cost_ratings5)
results = cursor.fetchall()

# Create a DataFrame from the results
df_cost_ratings = pd.DataFrame(results, columns=['approx_cost_for_two_people', 'rate'])

# Define price bins and labels. Adjust bins based on actual data distribution if needed.
bins = [0, 250, 500, 750, 1000, 1500, 2000, np.inf]
labels = ['<250', '250-500', '500-750', '750-1000', '1000-1500', '1500-2000', '>2000']

df_cost_ratings['price_range'] = pd.cut(df_cost_ratings['approx_cost_for_two_people'], bins=bins, labels=labels, right=False)

# Calculate average rating per price range
df_satisfaction_by_price = df_cost_ratings.groupby('price_range')['rate'].mean().reset_index()

# Sort by average rating in descending order
df_satisfaction_by_price = df_satisfaction_by_price.sort_values(by='rate', ascending=False)

# Display the results
print("Average Customer Satisfaction (Ratings) by Price Range:")
display(df_satisfaction_by_price)

Average Customer Satisfaction (Ratings) by Price Range:


C:\Users\Haritha\AppData\Local\Temp\ipykernel_2444\482807772.py:28: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_satisfaction_by_price = df_cost_ratings.groupby('price_range')['rate'].mean().reset_index()


,price_range,rate
5,1500-2000,4.230265
6,>2000,4.179316
4,1000-1500,4.083606
3,750-1000,3.941550
0,<250,3.883429
2,500-750,3.792309
1,250-500,3.785822


In [16]:
import pandas as pd

# SQL query to get the count of restaurants by cuisine
query_most_common_cuisines7 = """
SELECT
    cuisines,
    COUNT(name) AS restaurant_count
FROM
    ubereats
GROUP BY
    cuisines
ORDER BY
    restaurant_count DESC
LIMIT 10;
"""

# Execute the query and fetch results
cursor.execute(query_most_common_cuisines7)
results_cuisines = cursor.fetchall()

# Get column names from cursor description
column_names_cuisines = [desc[0] for desc in cursor.description]

# Create a DataFrame from the results
df_most_common_cuisines = pd.DataFrame(results_cuisines, columns=column_names_cuisines)

# Display the top 10 most common cuisines
print("Top 10 Most Common Cuisines in Bangalore:")
display(df_most_common_cuisines)

Top 10 Most Common Cuisines in Bangalore:


,cuisines,restaurant_count
0,North Indian,1144
1,"North Indian, Chinese",776
2,South Indian,359
3,Cafe,273
4,"South Indian, North Indian, Chinese",233
5,"Bakery, Desserts",215
6,"Desserts, Beverages",214
7,Chinese,210
8,"Ice Cream, Desserts",208
9,Desserts,206


In [17]:
import pandas as pd

# SQL query to get the average rate by cuisine
query_highest_rated_cuisines8 = """
SELECT
    cuisines,
    AVG(rate) AS average_rating
FROM
    ubereats
GROUP BY
    cuisines
ORDER BY
    average_rating DESC
LIMIT 10;
"""

# Execute the query and fetch results
cursor.execute(query_highest_rated_cuisines8)
results_rated_cuisines = cursor.fetchall()

# Get column names from cursor description
column_names_rated_cuisines = [desc[0] for desc in cursor.description]

# Create a DataFrame from the results
df_highest_rated_cuisines = pd.DataFrame(results_rated_cuisines, columns=column_names_rated_cuisines)

# Display the top 10 highest-rated cuisines
print("Top 10 Cuisines with the Highest Average Ratings:")
display(df_highest_rated_cuisines)

Top 10 Cuisines with the Highest Average Ratings:


,cuisines,average_rating
0,"Healthy Food, Salad, Mediterranean",4.900000
1,"Asian, Chinese, Thai, Momos",4.900000
2,"Continental, North Indian, Italian, South Indi...",4.900000
3,"North Indian, European, Mediterranean, BBQ",4.800000
4,"Asian, Mediterranean, North Indian, BBQ",4.800000
5,"European, Mediterranean, North Indian, BBQ",4.789474
6,"American, Tex-Mex, Burger, BBQ, Mexican",4.750000
7,"Continental, Asian, North Indian",4.723809
8,"North Indian, European, Mediterranean",4.700000
9,"Sushi, Japanese, Chinese, Thai",4.700000


In [18]:

# SQL query to get average rate and restaurant count for all locations
query_location_insights11 = """
SELECT
    location,
    AVG(rate) AS average_rating,
    COUNT(name) AS restaurant_count
FROM
    ubereats
GROUP BY
    location;
"""

# Execute the query and fetch results
cursor.execute(query_location_insights11)
results_location_insights = cursor.fetchall()

# Get column names from cursor description
column_names_location_insights = [desc[0] for desc in cursor.description]

# Create a DataFrame from the results
df_location_insights = pd.DataFrame(results_location_insights, columns=column_names_location_insights)

# Sort by average rating (descending) and then by restaurant count (ascending)
# This helps identify highly-rated but less saturated locations
df_ideal_locations = df_location_insights.sort_values(by=['average_rating', 'restaurant_count'],
                                                        ascending=[False, True])

print("Ideal Locations for Premium Restaurant Onboarding (sorted by average rating desc, then restaurant count asc):")
display(df_ideal_locations.head(10))

Ideal Locations for Premium Restaurant Onboarding (sorted by average rating desc, then restaurant count asc):


,location,average_rating,restaurant_count
79,Lavelle Road,4.192534,442
19,Koramangala 5th Block,4.148312,1782
10,Sankey Road,4.105882,17
14,Koramangala 3rd Block,4.101242,161
76,Cunningham Road,4.100300,333
80,St. Marks Road,4.099013,304
28,Koramangala 2nd Block,4.073333,45
70,Sadashiv Nagar,4.055263,38
33,Residency Road,4.051364,440
37,Church Street,4.045455,506


In [19]:
query_both_services = """
SELECT
    online_order,
    book_table,
    AVG(rate) AS average_rating
FROM
    ubereats
GROUP BY
    online_order, book_table
ORDER BY
    average_rating DESC;
"""

# Execute the query and fetch results
cursor.execute(query_both_services)
results_both_services = cursor.fetchall()

# Get column names from cursor description
column_names_both_services = [desc[0] for desc in cursor.description]

# Create a DataFrame from the results
df_both_services = pd.DataFrame(results_both_services, columns=column_names_both_services)

# Display the results
print("Average Restaurant Ratings by Online Ordering and Table Booking Availability:")
display(df_both_services)

Average Restaurant Ratings by Online Ordering and Table Booking Availability:


,online_order,book_table,average_rating
0,No,Yes,4.175261
1,Yes,Yes,4.144309
2,Yes,No,3.821718
3,No,No,3.789804


In [ ]:
import pandas as pd
import numpy as np

# SQL query to fetch restaurant name, approximate cost, and rate

query_restaurant_performance = """
SELECT 
name,rate, approx_cost_for_two_people
FROM
    ubereats
WHERE 
rate IS NOT NULL;
"""

# Execute the query and fetch results
cursor.execute(query_restaurant_performance)
results_restaurant_performance = cursor.fetchall()

# Get column names from cursor description
column_names_performance = [desc[0] for desc in cursor.description]

# Create a DataFrame from the results
df_restaurant_performance = pd.DataFrame(results_restaurant_performance, columns=column_names_performance)


In [23]:
import pandas as pd
filepath= r"C:\Users\Haritha\Downloads\orders.json"
a=pd.read_json(filepath)
display(a)

,order_id,restaurant_name,order_date,order_value,discount_used,payment_method
0,8174c2af-07a4-4837-9068-1169d963e36e,TBC Sky Lounge,2025-05-06,201.87,No,Card
1,0f9ddb57-4632-4a9f-9afe-1d2e41b94e32,KC Das - Sweet Paradise,2026-01-08,1392.27,No,Cash
2,099deda6-8c53-41cb-abf8-00126e6b2643,Hotel Kadamba Veg,2026-01-22,1358.35,Yes,UPI
3,94a6ba27-d6ef-4d10-a026-ac0c0afa8505,Cafe Srinidhi,2025-11-10,782.37,No,UPI
4,216f3ec7-80a9-4d95-872f-715c37163b6d,Pebble,2025-12-22,1551.09,Yes,UPI
...,...,...,...,...,...,...
24995,74d3cd31-c1e0-4331-aad7-e1f48b4698ae,Carnatic,2025-11-24,1254.91,No,UPI
24996,14db2f6b-f722-44a2-ae55-fe42511e4bb2,Reddy's Restaurant,2025-06-24,760.39,No,Card
24997,0fd85321-c1ce-4163-beca-c8ed2b472b8d,BTM Bawarchi Biryani,2025-12-04,707.99,Yes,Card
24998,6b96c699-2d63-4a3c-84f1-5d52ca4bf357,Nite Out,2025-09-21,724.29,Yes,Card


In [27]:
a.drop_duplicates()

,order_id,restaurant_name,order_date,order_value,discount_used,payment_method
0,8174c2af-07a4-4837-9068-1169d963e36e,TBC Sky Lounge,2025-05-06,201.87,No,Card
1,0f9ddb57-4632-4a9f-9afe-1d2e41b94e32,KC Das - Sweet Paradise,2026-01-08,1392.27,No,Cash
2,099deda6-8c53-41cb-abf8-00126e6b2643,Hotel Kadamba Veg,2026-01-22,1358.35,Yes,UPI
3,94a6ba27-d6ef-4d10-a026-ac0c0afa8505,Cafe Srinidhi,2025-11-10,782.37,No,UPI
4,216f3ec7-80a9-4d95-872f-715c37163b6d,Pebble,2025-12-22,1551.09,Yes,UPI
...,...,...,...,...,...,...
24995,74d3cd31-c1e0-4331-aad7-e1f48b4698ae,Carnatic,2025-11-24,1254.91,No,UPI
24996,14db2f6b-f722-44a2-ae55-fe42511e4bb2,Reddy's Restaurant,2025-06-24,760.39,No,Card
24997,0fd85321-c1ce-4163-beca-c8ed2b472b8d,BTM Bawarchi Biryani,2025-12-04,707.99,Yes,Card
24998,6b96c699-2d63-4a3c-84f1-5d52ca4bf357,Nite Out,2025-09-21,724.29,Yes,Card


In [28]:
a.isnull().sum()

order_id           0
restaurant_name    0
order_date         0
order_value        0
discount_used      0
payment_method     0
dtype: int64

In [32]:
table_name = 'orders_data'

drop_table_sql = f"DROP TABLE IF EXISTS {table_name}"
cursor.execute(drop_table_sql)

create_table_sql = f"""CREATE TABLE {table_name} (
    order_id VARCHAR(255) PRIMARY KEY,
    restaurant_name VARCHAR(255),
    order_date DATE,
    order_value DECIMAL(10, 2),
    discount_used BOOLEAN,
    payment_method VARCHAR(50)
)"""
cursor.execute(create_table_sql)
print(f"Table '{table_name}' created successfully.")

Table 'orders_data' created successfully.


In [33]:
from datetime import datetime

# Convert order_date to datetime objects and discount_used to tinyint (0/1)
df_to_insert = a.copy()
df_to_insert['order_date'] = pd.to_datetime(df_to_insert['order_date']).dt.date
df_to_insert['discount_used'] = df_to_insert['discount_used'].apply(lambda x: 1 if x == 'Yes' else 0)

insert_sql = f"""INSERT INTO {table_name} (
    order_id, restaurant_name, order_date, order_value, discount_used, payment_method
) VALUES (
    %s, %s, %s, %s, %s, %s
)"""

# Prepare data as a list of tuples
records_to_insert = df_to_insert.values.tolist()

# Insert records in batches (optional, but good for performance)
batch_size = 1000
for i in range(0, len(records_to_insert), batch_size):
    batch = records_to_insert[i:i + batch_size]
    cursor.executemany(insert_sql, batch)
    conn.commit() # Commit after each batch

print(f"Successfully inserted {len(records_to_insert)} records into '{table_name}'.")

Successfully inserted 25000 records into 'orders_data'.


In [34]:
select_sql = f"SELECT * FROM {table_name} LIMIT 5"
cursor.execute(select_sql)
result = cursor.fetchall()

# Print column names
columns = [i[0] for i in cursor.description]
print(columns)

# Print fetched rows
for row in result:
    print(row)

cursor.close()
conn.close()
print("Connection closed.")

['order_id', 'restaurant_name', 'order_date', 'order_value', 'discount_used', 'payment_method']
('000165ab-4daa-4af3-bbc0-87fe2e9410f4', 'Lakshmi Natraja Refreshments', datetime.date(2025, 8, 2), Decimal('616.27'), 0, 'UPI')
('000f8bdf-aa32-4346-a1c9-eb1573c1029c', 'Cubbonpete Donne Biryani', datetime.date(2025, 7, 15), Decimal('655.91'), 1, 'Cash')
('00107261-cee1-4e43-b2d2-1f46edb9464a', 'Ashirvaad Grand', datetime.date(2025, 7, 17), Decimal('1196.35'), 1, 'Cash')
('0013ed33-0a88-4d80-9f6b-d0282f3710dd', 'Sri Venkateshwara Sweet Meat Stall', datetime.date(2026, 1, 5), Decimal('1194.37'), 0, 'UPI')
('00154192-5ca4-4546-abf2-de488483d1f0', 'The Dining Table - Sahar Pavillion', datetime.date(2026, 1, 17), Decimal('550.19'), 1, 'UPI')
Connection closed.


In [ ]:

# --- Question 1: What is the total order value for each restaurant?
question_1 = "What is the total order value for each restaurant?"
sql_query_1 = f"""SELECT restaurant_name, SUM(order_value) AS total_revenue
                  FROM orders_data
                  GROUP BY restaurant_name
                  ORDER BY total_revenue DESC;"""

print(f"\nQuestion 1: {question_1}")
print(f"SQL Query 1:\n{sql_query_1}")
cursor.execute(sql_query_1)
result_1 = cursor.fetchall()
columns_1 = [i[0] for i in cursor.description]
print(f"Result 1 (Top 5):\n{columns_1}")
for row in result_1[:10]: 
    print(row)


Question 1: What is the total order value for each restaurant?
SQL Query 1:
SELECT restaurant_name, SUM(order_value) AS total_revenue
                  FROM orders_data
                  GROUP BY restaurant_name
                  ORDER BY total_revenue DESC;
Result 1 (Top 5):
['restaurant_name', 'total_revenue']
("Bob's Bar", Decimal('19518.14'))
('Cake Cafe', Decimal('19335.17'))
('Biryani Mane', Decimal('19249.45'))
('Hungry Lee', Decimal('19167.53'))
('Andhra Grills', Decimal('19153.72'))
('Mighty Paws', Decimal('18410.50'))
('Delhi Ke Bawarchi', Decimal('18131.68'))
('Pingara', Decimal('17954.06'))
('Bawarchi Inn', Decimal('17717.43'))
("Soup'ermanz Kitchen", Decimal('17350.72'))


In [45]:
# --- Question 2: Which payment method is most frequently used?
question_2 = "Which payment method is most frequently used?"
sql_query_2 = f"""SELECT payment_method, COUNT(order_id) AS number_of_orders
                  FROM orders_data
                  GROUP BY payment_method
                  ORDER BY number_of_orders DESC;"""

print(f"\nQuestion 2: {question_2}")
print(f"SQL Query 2:\n{sql_query_2}")
cursor.execute(sql_query_2)
result_2 = cursor.fetchall()
columns_2 = [i[0] for i in cursor.description]
print(f"Result 2:\n{columns_2}")
for row in result_2:
    print(row)


Question 2: Which payment method is most frequently used?
SQL Query 2:
SELECT payment_method, COUNT(order_id) AS number_of_orders
                  FROM orders_data
                  GROUP BY payment_method
                  ORDER BY number_of_orders DESC;
Result 2:
['payment_method', 'number_of_orders']
('Cash', 8384)
('Card', 8364)
('UPI', 8252)


In [46]:
# --- Question 3: What is the average order value across all orders?
question_3 = "What is the average order value across all orders?"
sql_query_3 = f"""SELECT AVG(order_value) AS average_order_value
                  FROM {table_name};"""

print(f"\nQuestion 3: {question_3}")
print(f"SQL Query 3:\n{sql_query_3}")
cursor.execute(sql_query_3)
result_3 = cursor.fetchone()
columns_3 = [i[0] for i in cursor.description]
print(f"Result 3:\n{columns_3}")
print(result_3)



Question 3: What is the average order value across all orders?
SQL Query 3:
SELECT AVG(order_value) AS average_order_value
                  FROM orders_data;
Result 3:
['average_order_value']
(Decimal('986.081873'),)


In [47]:
# --- Question 4: How many orders used a discount vs. no discount?
question_4 = "How many orders used a discount vs. no discount?"
sql_query_4 = f"""SELECT
                    CASE
                        WHEN discount_used = 1 THEN 'Yes'
                        ELSE 'No'
                    END AS discount_status,
                    COUNT(order_id) AS number_of_orders
                  FROM {table_name}
                  GROUP BY discount_status;"""

print(f"\nQuestion 4: {question_4}")
print(f"SQL Query 4:\n{sql_query_4}")
cursor.execute(sql_query_4)
result_4 = cursor.fetchall()
columns_4 = [i[0] for i in cursor.description]
print(f"Result 4:\n{columns_4}")
for row in result_4:
    print(row)


Question 4: How many orders used a discount vs. no discount?
SQL Query 4:
SELECT
                    CASE
                        WHEN discount_used = 1 THEN 'Yes'
                        ELSE 'No'
                    END AS discount_status,
                    COUNT(order_id) AS number_of_orders
                  FROM orders_data
                  GROUP BY discount_status;
Result 4:
['discount_status', 'number_of_orders']
('No', 12509)
('Yes', 12491)


In [42]:
# --- Question 5: Which are the top 5 restaurants by total number of orders?
question_5 = "Which are the top 5 restaurants by total number of orders?"
sql_query_5 = f"""SELECT restaurant_name, COUNT(order_id) AS total_orders
                  FROM {table_name}
                  GROUP BY restaurant_name
                  ORDER BY total_orders DESC
                  LIMIT 5;"""

print(f"\nQuestion 5: {question_5}")
print(f"SQL Query 5:\n{sql_query_5}")
cursor.execute(sql_query_5)
result_5 = cursor.fetchall()
columns_5 = [i[0] for i in cursor.description]
print(f"Result 5:\n{columns_5}")
for row in result_5:
    print(row)



Question 5: Which are the top 5 restaurants by total number of orders?
SQL Query 5:
SELECT restaurant_name, COUNT(order_id) AS total_orders
                  FROM orders_data
                  GROUP BY restaurant_name
                  ORDER BY total_orders DESC
                  LIMIT 5;
Result 5:
['restaurant_name', 'total_orders']
('Andhra Grills', 19)
('Cake Cafe', 19)
('Shree Muthahalli Veg', 18)
('Orbis Restaurant', 18)
('Hungry Lee', 18)
